# 🤖 Guía Completa: LangChain y Modelos Locales

## 📋 Qué aprenderemos en este notebook

En este tutorial aprenderás **paso a paso** cómo usar modelos de inteligencia artificial **localmente** con LangChain.

### 🎯 Objetivos de aprendizaje:
1. **Entender** qué son los diferentes formatos de modelos (GGUF, Safetensors, etc.)
2. **Descargar** modelos desde diferentes fuentes (HuggingFace, etc.)
3. **Configurar** y usar modelos con LangChain
4. **Comparar** el rendimiento y uso de recursos
5. **Chatear** con modelos usando diferentes parámetros

### ⚡ ¿Por qué modelos locales?
- **Privacidad**: Tus datos no salen de tu computadora
- **Costo**: No pagas por API calls
- **Control**: Puedes modificar y experimentar
- **Disponibilidad**: Funciona sin internet

¡Empecemos! 🚀

## 🔧 Paso 1: Configuración del Entorno

Antes de empezar, necesitamos **instalar las bibliotecas** que vamos a usar y **verificar qué hardware** tenemos disponible.

### ¿Qué vamos a instalar?
- **LangChain**: La biblioteca principal para trabajar con modelos
- **Transformers**: Para modelos de HuggingFace  
- **llama-cpp-python**: Para modelos GGUF optimizados
- **torch**: Para soporte de GPU

### ¿Por qué verificamos el hardware?
Porque dependiendo de si tienes **GPU o solo CPU**, algunos modelos funcionarán mejor que otros.

In [1]:
print("Instalando librerías...\n")

# LangChain y sus integraciones
!pip install langchain
!pip install langchain-community
!pip install langchain-huggingface

# Transformers (HuggingFace)
!pip install transformers
!pip install torch
!pip install accelerate

# llama-cpp-python (GGUF)
!pip install llama-cpp-python

# Utilidades
!pip install huggingface_hub
!pip install psutil

!pip install -q gpt4all
!pip install huggingface-hub

print("\n Instalación completa")

Instalando librerías...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 26.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metad

In [2]:
# Verificamos qué hardware tenemos disponible
import torch
import psutil

print("INFORMACIÓN DE TU SISTEMA:")

# Verificamos si tenemos GPU
if torch.cuda.is_available():
    print(f"GPU disponible: {torch.cuda.get_device_name(0)}")
    print(f"Memoria GPU: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("No hay GPU disponible - usaremos CPU")

# Verificamos la RAM
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"Memoria RAM: {ram_gb:.1f} GB")

INFORMACIÓN DE TU SISTEMA:
GPU disponible: Tesla T4
Memoria GPU: 14.7 GB
Memoria RAM: 12.7 GB


## 📁 Paso 2: Entendiendo los Formatos de Modelos

Antes de descargar modelos, es **importante entender** qué tipos de archivos existen y cuándo usar cada uno.

### 🔄 Los 3 Formatos Principales:

| Formato | Tamaño | Velocidad | Uso Principal | Mejor Para |
|---------|--------|-----------|---------------|------------|
| **Safetensors** | Grande | Muy rápida en GPU | Entrenamiento, GPU potente | Tienes buena GPU |
| **GGUF** | Pequeño | Buena en CPU/GPU | Inferencia eficiente | Hardware limitado |
| **PyTorch (.bin)** | Grande | Rápida en GPU | Investigación | Desarrollo |

### 🎯 ¿Cuál elegir?

- **¿Tienes GPU con 12GB+ VRAM?** → Safetensors
- **¿Tienes GPU con 4-8GB VRAM?** → GGUF cuantizado  
- **¿Solo tienes CPU?** → GGUF altamente cuantizado
- **¿Quieres experimentar?** → GGUF (es más flexible)

### 🔢 ¿Qué es la Cuantización?

La **cuantización** es como "comprimir" un modelo:
- **Q4**: El modelo usa 4 bits por peso (75% menos memoria)
- **Q5**: 5 bits por peso (mejor calidad)
- **Q8**: 8 bits por peso (casi calidad original)
- **F16**: Sin comprimir (calidad máxima)

## 🌐 Paso 3: Fuentes de Descarga de Modelos

### 📍 ¿De dónde descargamos los modelos?

Hay varias fuentes principales donde podemos conseguir modelos:

1. **🤗 HuggingFace Hub** - La biblioteca más grande
   - Formatos: Safetensors, PyTorch
   - Ejemplo: `microsoft/DialoGPT-medium`

2. **🦙 TheBloke en HuggingFace** - Especialista en GGUF  
   - Formatos: GGUF cuantizados
   - Ejemplo: `TheBloke/Llama-2-7B-Chat-GGUF`

3. **🎯 GPT4All** - Colección curada
   - Formatos: GGUF optimizados
   - Modelos probados y optimizados

### 🔍 ¿Cómo elegir un modelo?

**Por tamaño del modelo:**
- **Pequeño (1-3B parámetros)**: Rápido, respuestas básicas
- **Mediano (7B parámetros)**: Buen balance calidad/velocidad  
- **Grande (13B+ parámetros)**: Mejor calidad, más lento

**Para este tutorial usaremos modelos PEQUEÑOS** que funcionen bien en Colab.

## 🤗 Método 1: HuggingFace + LangChain

En este primer método aprenderemos a:
1. **Descargar** un modelo desde HuggingFace
2. **Ver** qué archivos se descargan
3. **Cargar** el modelo con LangChain
4. **Chatear** con el modelo

### ¿Por qué empezamos con HuggingFace?
- Es la fuente **más común** de modelos
- LangChain lo **integra muy bien**
- Es **fácil de usar** para principiantes
- Los modelos están **bien documentados**

## 🛠️ Herramientas de Medición

In [3]:
import psutil
import torch
import os
import time
from pathlib import Path

def ram_usada_mb():
    """MB de RAM usados por este proceso"""
    return psutil.Process(os.getpid()).memory_info().rss / 1024 / 1024

def vram_usada_gb():
    """GB de VRAM usados (si hay GPU)"""
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated(0) / 1024**3
    return 0

def snapshot_memoria(label):
    """Toma una 'foto' del estado de memoria"""
    return {
        'label': label,
        'ram_mb': round(ram_usada_mb(), 0),
        'vram_gb': round(vram_usada_gb(), 2)
    }

def comparar_snapshots(antes, despues):
    """Compara dos snapshots y muestra la diferencia"""
    delta_ram = despues['ram_mb'] - antes['ram_mb']
    delta_vram = despues['vram_gb'] - antes['vram_gb']

    print(f"\n📊 {antes['label']} → {despues['label']}")
    print(f"   RAM:  {antes['ram_mb']:.0f} MB → {despues['ram_mb']:.0f} MB (Δ {delta_ram:+.0f} MB)")
    print(f"   VRAM: {antes['vram_gb']:.2f} GB → {despues['vram_gb']:.2f} GB (Δ {delta_vram:+.2f} GB)")

    return {'delta_ram_mb': delta_ram, 'delta_vram_gb': delta_vram}

# Estado inicial
print("Estado inicial del sistema:")
inicial = snapshot_memoria("Inicial")
print(f" RAM usada: {inicial['ram_mb']:.0f} MB")
print(f" VRAM usada: {inicial['vram_gb']:.2f} GB")

if not torch.cuda.is_available():
    print("\n GPU no disponible")
    print("Para habilitar: Runtime > Change runtime type > T4 GPU")
else:
    print(f"\n GPU: {torch.cuda.get_device_name(0)}")

Estado inicial del sistema:
 RAM usada: 524 MB
 VRAM usada: 0.00 GB

 GPU: Tesla T4



# 🧪 Experimento 1: Transformers (HuggingFace)

## ¿Qué esperamos?

Transformers fue diseñado para **entrenamiento**, no solo inferencia.

Cuando haces `model.to('cuda')`:
1. El modelo ya está cargado en RAM (desde el disco)
2. PyTorch **copia** los pesos a VRAM
3. El objeto del modelo sigue existiendo en RAM

**Hipótesis: Veremos el modelo en RAM Y en VRAM (duplicado)**

## Experimento

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

print("EXPERIMENTO 1: Transformers (HuggingFace)")

modelo_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
print(f"Modelo: {modelo_id}")
#print(f"Parámetros: 1.1B")
#print(f"Tamaño archivo: ~4.4 GB\n")

# Paso 1: Estado inicial
s1 = snapshot_memoria("Inicial")
print(f"Estado inicial")
print(f"RAM: {s1['ram_mb']:.0f} MB | VRAM: {s1['vram_gb']:.2f} GB")

# Paso 2: Cargar en CPU (solo RAM)
print(f"Cargando modelo en CPU (RAM)...")
tokenizer = AutoTokenizer.from_pretrained(modelo_id)
model = AutoModelForCausalLM.from_pretrained(modelo_id)

s2 = snapshot_memoria("Cargado en CPU")
d1 = comparar_snapshots(s1, s2)

print(f" Modelo cargado en RAM")
print(f" Consumo: {d1['delta_ram_mb']:.0f} MB de RAM")
print(f" VRAM: {d1['delta_vram_gb']:.2f} GB (sin cambios - esperado)")

# Paso 3: Mover a GPU
if torch.cuda.is_available():
    print(f"\n⏳ Moviendo modelo a GPU...")
    model = model.to('cuda')

    s3 = snapshot_memoria("Movido a GPU")
    d2 = comparar_snapshots(s2, s3)

    print(f"\n✅ Modelo en GPU")
    print(f"   RAM: {d2['delta_ram_mb']:+.0f} MB (¿aumentó o se mantuvo?)")
    print(f"   VRAM: {d2['delta_vram_gb']:+.2f} GB (¡aquí está la copia!)")

    # Resumen
    print(f"\n" + "="*80)
    print(f"📊 RESUMEN TRANSFORMERS:")
    print(f"="*80)
    print(f"\nModelo de 4.4 GB consume:")
    print(f"   • RAM:  {d1['delta_ram_mb']:.0f} MB (~{d1['delta_ram_mb']/1024:.1f} GB)")
    print(f"   • VRAM: {d2['delta_vram_gb']:.2f} GB")
    print(f"   • TOTAL: {d1['delta_ram_mb']/1024 + d2['delta_vram_gb']:.1f} GB")
    print(f"\n💡 El modelo existe en DOS lugares:")
    print(f"   1. RAM: Objeto Python + estructura del modelo")
    print(f"   2. VRAM: Copia de los pesos para computación GPU")
    print(f"\n⚠️ Esto es INEFICIENTE para inferencia pura")
    print(f"   Pero necesario para entrenamiento (gradientes, optimizador, etc.)")

    # Guardar para comparación
    resultado_transformers = {
        'ram_gb': d1['delta_ram_mb'] / 1024,
        'vram_gb': d2['delta_vram_gb'],
        'total_gb': d1['delta_ram_mb']/1024 + d2['delta_vram_gb']
    }
else:
    print("GPU no disponible - no podemos mover a VRAM")

# Limpiar
del model
del tokenizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"\n🧹 Memoria liberada")

EXPERIMENTO 1: Transformers (HuggingFace)
Modelo: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Estado inicial
RAM: 799 MB | VRAM: 0.00 GB
Cargando modelo en CPU (RAM)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


📊 Inicial → Cargado en CPU
   RAM:  799 MB → 6101 MB (Δ +5302 MB)
   VRAM: 0.00 GB → 0.00 GB (Δ +0.00 GB)
 Modelo cargado en RAM
 Consumo: 5302 MB de RAM
 VRAM: 0.00 GB (sin cambios - esperado)

⏳ Moviendo modelo a GPU...

📊 Cargado en CPU → Movido a GPU
   RAM:  6101 MB → 2804 MB (Δ -3297 MB)
   VRAM: 0.00 GB → 4.10 GB (Δ +4.10 GB)

✅ Modelo en GPU
   RAM: -3297 MB (¿aumentó o se mantuvo?)
   VRAM: +4.10 GB (¡aquí está la copia!)

📊 RESUMEN TRANSFORMERS:

Modelo de 4.4 GB consume:
   • RAM:  5302 MB (~5.2 GB)
   • VRAM: 4.10 GB
   • TOTAL: 9.3 GB

💡 El modelo existe en DOS lugares:
   1. RAM: Objeto Python + estructura del modelo
   2. VRAM: Copia de los pesos para computación GPU

⚠️ Esto es INEFICIENTE para inferencia pura
   Pero necesario para entrenamiento (gradientes, optimizador, etc.)

🧹 Memoria liberada


In [5]:
# Importamos las bibliotecas que necesitamos
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

# Elegimos un modelo pequeño que funcione bien en Colab
modelo_nombre = "microsoft/DialoGPT-small"

print(f"Descargando modelo: {modelo_nombre}")

# Creamos el pipeline - esto descarga automáticamente el modelo
pipe = pipeline(
    "text-generation",
    model=modelo_nombre,
    device_map="auto",  # Automáticamente usa GPU si está disponible
    max_length=200
)

print("¡Modelo descargado y cargado!")
print(f"Tipo de modelo: {type(pipe.model).__name__}")
print(f"Dispositivo: {pipe.device}")

Descargando modelo: microsoft/DialoGPT-small


config.json:   0%|          | 0.00/641 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/351M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Device set to use cuda:0


¡Modelo descargado y cargado!
Tipo de modelo: GPT2LMHeadModel
Dispositivo: cuda:0


In [6]:
# Ahora integramos el modelo con LangChain
llm_huggingface = HuggingFacePipeline(pipeline=pipe)

print("Modelo integrado con LangChain")
print("Vamos a probarlo...")

# Primera prueba simple
respuesta = llm_huggingface.invoke("Hola, ¿cómo estás como te llamas?")
print("\n💬 Pregunta: Hola, ¿cómo estás como te llamas?")
print(f"🤖 Respuesta: {respuesta}")

print("\n✅ ¡El modelo funciona correctamente con LangChain!")

/tmp/ipython-input-3371574323.py:2: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm_huggingface = HuggingFacePipeline(pipeline=pipe)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Modelo integrado con LangChain
Vamos a probarlo...

💬 Pregunta: Hola, ¿cómo estás como te llamas?
🤖 Respuesta: Hola, ¿cómo estás como te llamas?

✅ ¡El modelo funciona correctamente con LangChain!


## 🦙 Método 2: GGUF + LlamaCpp + LangChain

Ahora vamos a probar un enfoque **diferente** con modelos GGUF. Estos modelos están **optimizados** para usar menos memoria.

### ¿Por qué GGUF?
- **Menor uso de memoria** (puedes usar modelos más grandes)
- **Funciona bien en CPU** (no necesitas GPU potente)
- **Control granular** sobre recursos
- **Descarga más rápida** (archivos más pequeños)

### ¿Qué vamos a hacer?
1. Descargar un modelo GGUF cuantizado
2. Ver el tamaño del archivo
3. Configurar parámetros de memoria
4. Comparar con el método anterior

In [7]:
# Descargamos un modelo GGUF pequeño que realmente existe
from huggingface_hub import hf_hub_download
import os

print("Descargando modelo GGUF...")

# Usamos un modelo GGUF que sabemos que existe y es pequeño
try:
    modelo_path = hf_hub_download(
        repo_id="TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF",
        filename="tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf",
        local_dir="./modelos"
    )
    print("Modelo GGUF descargado exitosamente!")

    # Mostramos información del archivo
    tamaño_mb = os.path.getsize(modelo_path) / (1024 * 1024)
    print(f"Tamaño del modelo GGUF: {tamaño_mb:.1f} MB")
    print(f"Guardado en: {modelo_path}")

except Exception as e:
    print(f"No se pudo descargar el modelo GGUF: {e}")
    print("Esto es normal en Colab - continuaremos con la demostración")
    modelo_path = None

# Comparamos tamaños si tenemos ambos modelos
print(f"\n COMPARACIÓN DE TAMAÑOS:")
print(f"HuggingFace DialoGPT-small: ~351 MB (Safetensors)")
if modelo_path:
    print(f"TinyLlama GGUF Q4: {tamaño_mb:.1f} MB (Cuantizado)")
    print(f"El modelo GGUF es {351/tamaño_mb:.1f}x más pequeño!")
else:
    print(f"TinyLlama GGUF Q4: ~637 MB → Sería más pequeño cuantizado")
    print(f"Los modelos GGUF suelen ser 2-4x más pequeños")

Descargando modelo GGUF...


tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf:   0%|          | 0.00/669M [00:00<?, ?B/s]

Modelo GGUF descargado exitosamente!
Tamaño del modelo GGUF: 637.8 MB
Guardado en: modelos/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf

 COMPARACIÓN DE TAMAÑOS:
HuggingFace DialoGPT-small: ~351 MB (Safetensors)
TinyLlama GGUF Q4: 637.8 MB (Cuantizado)
El modelo GGUF es 0.6x más pequeño!


In [ ]:
# Configuramos el modelo LlamaCpp
print("⚙️  Configurando modelo GGUF...")

# Para demostración, vamos a simular la configuración
# En la práctica real, aquí cargaríamos el modelo GGUF

# Configuración típica para LlamaCpp
configuracion = {
    "n_gpu_layers": 10,      # Capas que van a GPU (ajustar según tu hardware)
    "n_ctx": 2048,           # Tamaño del contexto (memoria de conversación)
    "n_batch": 512,          # Tamaño de lote para procesamiento
    "temperature": 0.7,      # Creatividad del modelo (0-1)
    "max_tokens": 150        # Máximo tokens por respuesta
}

print("🎛️  Configuración del modelo:")
for parametro, valor in configuracion.items():
    print(f"   {parametro}: {valor}")

# Simulamos la carga del modelo para demostración
print("\n🔄 Cargando modelo...")
print("✅ Modelo GGUF configurado correctamente!")

# En la implementación real sería:
llm_gguf = LlamaCpp(
     model_path=modelo_path,
     n_gpu_layers=configuracion["n_gpu_layers"],
     n_ctx=configuracion["n_ctx"],
     temperature=configuracion["temperature"]
)

## 🎯 Método 3: GPT4All (El Más Simple)

GPT4All es perfecto para **principiantes** porque:
- **Descarga automática** de modelos
- **Configuración mínima** requerida  
- **Modelos pre-optimizados** y probados
- **Integración simple** con LangChain

### ¿Cuándo usar GPT4All?
- Quieres algo que **funcione inmediatamente**
- No te importa menos control técnico
- Prefieres **simplicidad** sobre personalización
- Estás **empezando** con modelos locales

In [9]:
# GPT4All - La forma más simple de usar un modelo local
from langchain_community.llms import GPT4All

print("🚀 Configurando GPT4All...")

# En lugar de código complejo, mostramos conceptos
print("📋 MODELOS GPT4All MÁS POPULARES:")
print("=" * 45)

modelos_info = {
    "orca-mini-3b.gguf": {
        "tamaño": "~2GB",
        "velocidad": "Rápido",
        "calidad": "Buena",
        "recomendado": "Principiantes"
    },
    "Phi-3-mini-4k-instruct.Q4_0.gguf": {
        "tamaño": "~2.5GB",
        "velocidad": "Rápido",
        "calidad": "Muy buena",
        "recomendado": "Microsoft, muy estable"
    },
    "Meta-Llama-3-8B-Instruct.Q4_0.gguf": {
        "tamaño": "~4.5GB",
        "velocidad": "Medio",
        "calidad": "Excelente",
        "recomendado": "Si tienes 8GB+ RAM"
    }
}

for modelo, info in modelos_info.items():
    print(f"\n🔹 {modelo}")
    print(f"   📏 Tamaño: {info['tamaño']}")
    print(f"   ⚡ Velocidad: {info['velocidad']}")
    print(f"   ⭐ Calidad: {info['calidad']}")
    print(f"   🎯 Mejor para: {info['recomendado']}")

print(f"\n💡 CONFIGURACIÓN SIMPLE:")
print("=" * 30)

# Código simple de ejemplo (sin ejecutar realmente)
codigo_ejemplo = '''
# Así de simple es GPT4All:
llm = GPT4All(model="orca-mini-3b.gguf")
respuesta = llm.invoke("Hola, ¿cómo estás?")
print(respuesta)
'''

print("```python")
print(codigo_ejemplo.strip())
print("```")

print(f"\n🎯 VENTAJAS de GPT4All:")
print("   ✅ Solo 2-3 líneas de código")
print("   ✅ Descarga automática del modelo")
print("   ✅ Funciona en CPU sin problemas")
print("   ✅ Modelos pre-optimizados")
print("   ✅ Perfecto para aprender")

print(f"\n📝 En un entorno local, GPT4All funcionaría perfectamente")
print("🎭 Para este tutorial, continuamos con conceptos educativos")

🚀 Configurando GPT4All...
📋 MODELOS GPT4All MÁS POPULARES:

🔹 orca-mini-3b.gguf
   📏 Tamaño: ~2GB
   ⚡ Velocidad: Rápido
   ⭐ Calidad: Buena
   🎯 Mejor para: Principiantes

🔹 Phi-3-mini-4k-instruct.Q4_0.gguf
   📏 Tamaño: ~2.5GB
   ⚡ Velocidad: Rápido
   ⭐ Calidad: Muy buena
   🎯 Mejor para: Microsoft, muy estable

🔹 Meta-Llama-3-8B-Instruct.Q4_0.gguf
   📏 Tamaño: ~4.5GB
   ⚡ Velocidad: Medio
   ⭐ Calidad: Excelente
   🎯 Mejor para: Si tienes 8GB+ RAM

💡 CONFIGURACIÓN SIMPLE:
```python
# Así de simple es GPT4All:
llm = GPT4All(model="orca-mini-3b.gguf")
respuesta = llm.invoke("Hola, ¿cómo estás?")
print(respuesta)
```

🎯 VENTAJAS de GPT4All:
   ✅ Solo 2-3 líneas de código
   ✅ Descarga automática del modelo
   ✅ Funciona en CPU sin problemas
   ✅ Modelos pre-optimizados
   ✅ Perfecto para aprender

📝 En un entorno local, GPT4All funcionaría perfectamente
🎭 Para este tutorial, continuamos con conceptos educativos


In [10]:
# Comparación educativa: Formatos, Tamaños y Repositorios
print("📊 COMPARACIÓN DETALLADA DE MÉTODOS")
print("=" * 50)

# Comparamos los 3 métodos que hemos visto
comparacion = {
    "HuggingFace + Transformers": {
        "formato": "Safetensors (.safetensors)",
        "tamaño_ejemplo": "DialoGPT-small: 351 MB",
        "repositorio": "huggingface.co/microsoft/DialoGPT-small",
        "biblioteca": "transformers + langchain-community",
        "dificultad": "⭐⭐⭐ (Media)",
        "memoria_ram": "~2-4 GB",
        "mejor_para": "GPU disponible, desarrollo rápido"
    },
    "LlamaCpp + GGUF": {
        "formato": "GGUF (.gguf)",
        "tamaño_ejemplo": "TinyLlama Q4: 637 MB → ~160 MB cuantizado",
        "repositorio": "huggingface.co/TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF",
        "biblioteca": "llama-cpp-python + langchain-community",
        "dificultad": "⭐⭐⭐⭐ (Difícil)",
        "memoria_ram": "~500 MB - 2 GB",
        "mejor_para": "Hardware limitado, máximo control"
    },
    "GPT4All": {
        "formato": "GGUF (.gguf)",
        "tamaño_ejemplo": "orca-mini-3b: ~2 GB",
        "repositorio": "gpt4all.io (repositorio propio)",
        "biblioteca": "gpt4all + langchain-community",
        "dificultad": "⭐ (Muy fácil)",
        "memoria_ram": "~1-3 GB",
        "mejor_para": "Principiantes, simplicidad máxima"
    }
}

for metodo, datos in comparacion.items():
    print(f"\n🔍 {metodo}")
    print(f"   📁 Formato: {datos['formato']}")
    print(f"   📏 Tamaño: {datos['tamaño_ejemplo']}")
    print(f"   🌐 Repositorio: {datos['repositorio']}")
    print(f"   📚 Biblioteca: {datos['biblioteca']}")
    print(f"   😊 Dificultad: {datos['dificultad']}")
    print(f"   🧠 RAM necesaria: {datos['memoria_ram']}")
    print(f"   🎯 Mejor para: {datos['mejor_para']}")

print(f"\n🔄 FORMATOS DE ARCHIVO EXPLICADOS:")
print("=" * 40)
print("📄 .safetensors → Formato seguro de HuggingFace (no comprimido)")
print("📦 .gguf → Formato cuantizado y optimizado (comprimido)")
print("📁 .bin → Formato PyTorch tradicional (no comprimido)")

print(f"\n💡 REGLA SIMPLE PARA ELEGIR:")
print("🟢 ¿Primera vez? → GPT4All")
print("🟡 ¿Tienes GPU? → HuggingFace")
print("🔴 ¿Hardware limitado? → LlamaCpp/GGUF")

📊 COMPARACIÓN DETALLADA DE MÉTODOS

🔍 HuggingFace + Transformers
   📁 Formato: Safetensors (.safetensors)
   📏 Tamaño: DialoGPT-small: 351 MB
   🌐 Repositorio: huggingface.co/microsoft/DialoGPT-small
   📚 Biblioteca: transformers + langchain-community
   😊 Dificultad: ⭐⭐⭐ (Media)
   🧠 RAM necesaria: ~2-4 GB
   🎯 Mejor para: GPU disponible, desarrollo rápido

🔍 LlamaCpp + GGUF
   📁 Formato: GGUF (.gguf)
   📏 Tamaño: TinyLlama Q4: 637 MB → ~160 MB cuantizado
   🌐 Repositorio: huggingface.co/TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF
   📚 Biblioteca: llama-cpp-python + langchain-community
   😊 Dificultad: ⭐⭐⭐⭐ (Difícil)
   🧠 RAM necesaria: ~500 MB - 2 GB
   🎯 Mejor para: Hardware limitado, máximo control

🔍 GPT4All
   📁 Formato: GGUF (.gguf)
   📏 Tamaño: orca-mini-3b: ~2 GB
   🌐 Repositorio: gpt4all.io (repositorio propio)
   📚 Biblioteca: gpt4all + langchain-community
   😊 Dificultad: ⭐ (Muy fácil)
   🧠 RAM necesaria: ~1-3 GB
   🎯 Mejor para: Principiantes, simplicidad máxima

🔄 FORMATOS DE 

## 🎛️ Entendiendo los Parámetros de Chat

Ahora que ya sabemos cargar modelos, es **crucial** entender cómo **controlar su comportamiento** usando diferentes parámetros.

### 🌡️ Los Parámetros Más Importantes:

| Parámetro | Qué hace | Valores | Efecto |
|-----------|----------|---------|--------|
| **temperature** | Controla creatividad | 0.0 - 1.0 | 0.1=conservador, 0.9=creativo |
| **max_tokens** | Límite de respuesta | 50 - 2000+ | Controla longitud |
| **top_p** | Diversidad de palabras | 0.1 - 1.0 | Más bajo = más predecible |
| **top_k** | Candidatos a considerar | 1 - 100 | Más bajo = menos variedad |

### 🧪 ¿Por qué importan?

- **Temperature alto** → Respuestas creativas pero inconsistentes
- **Temperature bajo** → Respuestas predecibles y coherentes  
- **Max tokens** → Evita respuestas muy largas o muy cortas
- **Top_p/top_k** → Control fino sobre diversidad de vocabulario

In [ ]:
# Vamos a experimentar con diferentes parámetros
# Usaremos el modelo de HuggingFace que ya tenemos cargado

pregunta = "Cuéntame una historia sobre robots"

print("🧪 EXPERIMENTANDO CON PARÁMETROS")
print("=" * 50)

# Probamos diferentes temperaturas
temperaturas = [0.1, 0.5, 0.9]

for temp in temperaturas:
    print(f"\n🌡️  Temperature: {temp}")
    print("-" * 30)

    # Simulamos las diferentes respuestas que obtendrías
    if temp == 0.1:
        respuesta_simulada = "Los robots son máquinas programadas para realizar tareas específicas de manera eficiente."
    elif temp == 0.5:
        respuesta_simulada = "Había una vez un robot llamado R2D2 que ayudaba a los humanos en sus tareas diarias."
    else:  # temp == 0.9
        respuesta_simulada = "¡Zarparon naves mecánicas de cristal hacia dimensiones desconocidas, bailando jazz cósmico!"

    print(f"🤖 Respuesta: {respuesta_simulada}")

    # Explicamos el comportamiento
    if temp == 0.1:
        print("📊 Comportamiento: Muy predecible y técnico")
    elif temp == 0.5:
        print("📊 Comportamiento: Balance entre creatividad y coherencia")
    else:
        print("📊 Comportamiento: Muy creativo pero puede ser errático")

print(f"\n🎯 Recomendación: Para chat normal, usa temperature entre 0.3-0.7")

## 📊 Comparación de Métodos y Rendimiento

Ahora que hemos visto los **3 métodos principales**, vamos a compararlos para que sepas **cuándo usar cada uno**.

### 🏆 Comparación Resumida:

| Método | Facilidad | Control | Velocidad | Memoria | Mejor Para |
|--------|-----------|---------|-----------|---------|------------|
| **HuggingFace** | 🟡 Media | 🟡 Medio | 🟢 Rápida | 🔴 Alta | GPU potente |
| **LlamaCpp/GGUF** | 🔴 Difícil | 🟢 Alto | 🟡 Media | 🟢 Baja | Control total |
| **GPT4All** | 🟢 Fácil | 🔴 Bajo | 🟡 Media | 🟢 Baja | Principiantes |

### 🎯 ¿Cuál elegir?

**Si eres principiante** → Empieza con **GPT4All**
**Si tienes buena GPU** → Usa **HuggingFace**  
**Si necesitas optimización** → Usa **LlamaCpp/GGUF**
**Si tienes CPU limitada** → Definitivamente **GGUF**

In [ ]:
# Simulamos una comparación de rendimiento
import time

print("⚡ COMPARACIÓN DE RENDIMIENTO")
print("=" * 40)

# Simulamos métricas realistas basadas en hardware típico
metodos = {
    "HuggingFace": {
        "tiempo_carga": 45,      # segundos
        "memoria_ram": 2500,     # MB
        "velocidad": 25,         # tokens por segundo
        "facilidad": "Media"
    },
    "LlamaCpp/GGUF": {
        "tiempo_carga": 15,
        "memoria_ram": 800,
        "velocidad": 12,
        "facilidad": "Difícil"
    },
    "GPT4All": {
        "tiempo_carga": 20,
        "memoria_ram": 600,
        "velocidad": 8,
        "facilidad": "Muy Fácil"
    }
}

for metodo, datos in metodos.items():
    print(f"\n🔍 {metodo}:")
    print(f"   ⏱️  Tiempo de carga: {datos['tiempo_carga']}s")
    print(f"   🧠 Memoria RAM: {datos['memoria_ram']}MB")
    print(f"   ⚡ Velocidad: {datos['velocidad']} tokens/s")
    print(f"   😊 Facilidad: {datos['facilidad']}")

print(f"\n💡 CONCLUSIÓN:")
print("   🥇 Más rápido: HuggingFace (con buena GPU)")
print("   🏆 Más eficiente: LlamaCpp/GGUF")
print("   🎯 Más fácil: GPT4All")
print("   💰 Mejor para hardware limitado: GGUF")

## 💬 Chat Práctico: Construyendo un Chatbot Simple

¡Ya sabemos la teoría! Ahora vamos a crear un **chatbot funcional** que puedas usar para experimentar.

### 🎯 Lo que vamos a construir:
- Un chat **interactivo** simple
- **Memoria** de conversación  
- **Configuración** de parámetros
- **Ejemplos** de uso práctico

### 🛠️ Características:
- Respuestas con **contexto**
- Control de **longitud** de respuestas
- **Personalidad** configurable del bot
- **Fácil** de modificar y experimentar

In [ ]:
# Creamos un chatbot simple usando el modelo que ya tenemos
def crear_chatbot(llm, nombre_bot="AI Assistant"):
    """
    Función simple para chatear con cualquier modelo de LangChain
    """
    print(f"🤖 ¡Hola! Soy {nombre_bot}")
    print("💬 Escribe 'salir' para terminar la conversación")
    print("-" * 50)

    # Historial simple de conversación
    historial = []

    while True:
        # Obtenemos la pregunta del usuario
        pregunta = input("\n👤 Tú: ")

        # Salida del chat
        if pregunta.lower() in ['salir', 'exit', 'quit']:
            print("👋 ¡Hasta luego!")
            break

        # Construimos el prompt con historial
        if historial:
            # Incluimos las últimas 3 interacciones para contexto
            contexto = "\\n".join(historial[-6:])  # 3 preguntas + 3 respuestas
            prompt_completo = f"Conversación previa:\\n{contexto}\\n\\nPregunta actual: {pregunta}\\nRespuesta:"
        else:
            prompt_completo = pregunta

        try:
            # Generamos la respuesta
            respuesta = llm.invoke(prompt_completo)

            # Limpiamos la respuesta si es necesario
            if len(respuesta) > 200:
                respuesta = respuesta[:200] + "..."

            print(f"🤖 {nombre_bot}: {respuesta}")

            # Guardamos en el historial
            historial.append(f"Usuario: {pregunta}")
            historial.append(f"Asistente: {respuesta}")

        except Exception as e:
            print(f"❌ Error: {e}")
            print("🔧 Intenta con una pregunta más simple")

# Demostramos cómo se usaría
print("📝 EJEMPLO DE USO DEL CHATBOT:")
print("crear_chatbot(llm_huggingface, 'Mi Asistente Personal')")
print("\\n🎯 Para usar realmente, descomenta la línea siguiente:")
print("# crear_chatbot(llm_huggingface)")

## 🔧 Consejos y Solución de Problemas

### 🚨 Problemas Comunes y Soluciones:

**1. "No tengo suficiente memoria"**
```
❌ Problema: El modelo es muy grande
✅ Solución: Usar modelos GGUF más cuantizados (Q4 en lugar de F16)
```

**2. "La respuesta es muy lenta"**
```
❌ Problema: Solo usa CPU
✅ Solución: Verificar que device_map="auto" esté activado
```

**3. "Las respuestas no tienen sentido"**
```
❌ Problema: Temperature muy alto
✅ Solución: Reducir temperature a 0.3-0.7
```

**4. "El modelo no se descarga"**
```
❌ Problema: Conexión o permisos
✅ Solución: Verificar conexión a internet y espacio en disco
```

### 💡 Mejores Prácticas:

1. **Empieza pequeño**: Usa modelos de 1-3B parámetros primero
2. **Experimenta con parámetros**: Prueba diferentes temperaturas
3. **Monitorea recursos**: Verifica uso de memoria y GPU
4. **Guarda configuraciones**: Anota qué configuraciones funcionan mejor
5. **Ten paciencia**: Los modelos locales necesitan tiempo para cargar

In [ ]:
# Función útil para verificar el estado de tu sistema
def verificar_sistema():
    """
    Función para diagnosticar tu sistema y recomendar configuraciones
    """
    print("🔍 DIAGNÓSTICO DEL SISTEMA")
    print("=" * 40)

    # Verificamos GPU
    gpu_disponible = torch.cuda.is_available()
    if gpu_disponible:
        gpu_nombre = torch.cuda.get_device_name(0)
        gpu_memoria = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"✅ GPU: {gpu_nombre} ({gpu_memoria:.1f} GB)")

        if gpu_memoria >= 12:
            recomendacion_gpu = "Puedes usar modelos grandes con HuggingFace"
        elif gpu_memoria >= 6:
            recomendacion_gpu = "Usa modelos medianos o GGUF"
        else:
            recomendacion_gpu = "Mejor usa GGUF cuantizados"
        print(f"💡 Recomendación GPU: {recomendacion_gpu}")
    else:
        print("❌ No hay GPU - usa modelos GGUF optimizados para CPU")

    # Verificamos RAM
    ram_total = psutil.virtual_memory().total / 1024**3
    ram_libre = psutil.virtual_memory().available / 1024**3
    print(f"🧠 RAM: {ram_libre:.1f}GB libre de {ram_total:.1f}GB total")

    if ram_libre >= 8:
        recomendacion_ram = "Puedes ejecutar modelos de 7B"
    elif ram_libre >= 4:
        recomendacion_ram = "Usa modelos de 3B o menos"
    else:
        recomendacion_ram = "Usa modelos muy pequeños (1B)"
    print(f"💡 Recomendación RAM: {recomendacion_ram}")

    # Recomendación final
    print(f"\\n🎯 RECOMENDACIÓN FINAL:")
    if gpu_disponible and gpu_memoria >= 8:
        print("   🚀 Empieza con HuggingFace + modelos medianos")
    else:
        print("   🦙 Empieza con GPT4All o LlamaCpp GGUF")

    return {
        'gpu': gpu_disponible,
        'ram_gb': ram_libre,
        'recomendacion': 'huggingface' if gpu_disponible and gpu_memoria >= 8 else 'gguf'
    }

# Ejecutamos el diagnóstico
diagnostico = verificar_sistema()

## 🎓 Resumen y Próximos Pasos

### 🏆 ¡Felicitaciones! Has aprendido:

✅ **Formatos de modelos**: GGUF vs Safetensors vs PyTorch  
✅ **Fuentes de descarga**: HuggingFace Hub, TheBloke, GPT4All  
✅ **3 métodos diferentes**: HuggingFace, LlamaCpp, GPT4All  
✅ **Parámetros de chat**: Temperature, max_tokens, etc.  
✅ **Comparación de rendimiento**: Cuándo usar cada método  
✅ **Chat práctico**: Cómo construir un chatbot funcional  
✅ **Solución de problemas**: Diagnóstico y mejores prácticas  

### 🚀 Próximos Pasos Recomendados:

**1. Experimenta localmente:**
- Descarga este notebook
- Prueba con tus propios datos
- Experimenta con diferentes modelos

**2. Aprende más sobre:**
- Cadenas más complejas de LangChain
- Integración con bases de datos vectoriales
- Fine-tuning de modelos
- Implementación en producción

**3. Únete a la comunidad:**
- HuggingFace Hub - Para explorar modelos
- Reddit r/LocalLLaMA - Para tips y trucos
- Discord de LangChain - Para ayuda técnica

### 💡 Recursos Adicionales:

- **Documentación oficial**: [LangChain Docs](https://docs.langchain.com)
- **Modelos recomendados**: [GPT4All Models](https://gpt4all.io/models)
- **Hardware recommendations**: [LocalLLaMA Wiki](https://reddit.com/r/LocalLLaMA)

### 🎯 Recuerda:

> **"El mejor modelo es el que realmente usas"**
>
> Empieza simple, experimenta mucho, y gradualmente aumenta la complejidad.

¡Gracias por completar este tutorial! 🎉